# Kalama (GUIDE-1) — บันทึกความคืบหน้า: เฟส Automation

**Exploitation-Validated Vulnerability Prioritization**
ผู้ทำ: Natapat · อาจารย์ที่ปรึกษา: Atthapol
บันทึกวันที่: 2026-07-29

> notebook นี้สรุปว่าโปรเจกต์ถึงไหนแล้ว หลังจบเคสแรกแบบ manual และเริ่มออกแบบเฟส automation
> เก็บไว้อ่านทบทวน / ใช้เป็นฐานคุยกับ Atthapol รอบถัดไป


## 1. สถานะโดยรวม

| ส่วน | สถานะ |
|---|---|
| Full manual loop (CVE-2017-5645) | ✅ เสร็จ — Atthapol ให้ผ่านแล้ว |
| โจทย์ใหม่จาก Atthapol | ทำ pipeline ให้เป็น **automation** (โดยเฉพาะเรื่องการอัพเดท/เตรียม environment) |
| การออกแบบ automation | 🔄 กำลังคิด design — ยังไม่ลงมือเขียนโค้ดจริง |

**สิ่งที่ตกลงกันแล้วในเฟสนี้:** วิธี resolve dependency (hybrid source), การใช้ CPE+FixedVersion จาก Trivy, และ scope เฟสแรก
**สิ่งที่ยังเปิดอยู่:** การแยก "fixed version ผิด" vs "fix ไม่ได้ผล" (ข้อ 4 ด้านล่าง) และการตรวจ output จริงของ 4 CVE ที่เหลือ


## 2. ส่วนที่เสร็จแล้ว — CVE-2017-5645 (baseline reference)

Log4j TcpSocketServer Java deserialization RCE (CWE-502, CVSS 9.8)
**ไม่ใช่** Log4Shell/JNDI — เป็นคนละกลไก

Loop ที่ทำครบแบบ manual:

```
Trivy scan (oracle)  →  before-patch exploit  →  patch  →  re-exploit
   พบ CVE-2017-5645       ✅ /tmp/success        2.8.1→2.8.2   ❌ ไม่มี /tmp/success
   FixedVersion=2.8.2         ถูกสร้าง                            = patch เวิร์คจริง
```

**ข้อสรุปเชิงวิจัยจากเคสนี้:** FixedVersion (2.8.2) ที่ Trivy ระบุ *ยืนยันได้จริง* ด้วยการ exploit — เป็น True Positive
แต่เคสนี้ "สะอาดผิดปกติ" (version bump แบบ drop-in) ซึ่งจะเห็นด้านล่างว่า CVE อื่นไม่ง่ายเท่านี้


## 3. โจทย์เฟสใหม่ — และปัญหาจริงที่ทำให้มันยาก

Atthapol อยากให้ automate การเตรียม/อัพเดท environment
**แต่ปัญหาจริงไม่ใช่ "automate exploit"** (อันนั้นเป็น script ธรรมดา) — ปัญหาคือ **environment มันเน่าตามเวลา** แล้วต้อง manual debug ทุกครั้ง

เคสที่เจอจริงตอนทำ manual แบ่งได้ 2 ประเภท:

| Type | ตัวอย่างที่เจอ | ธรรมชาติ |
|---|---|---|
| **A — resource หาย/ถูกถอด** | `openjdk:8-jre-slim` ถูกถอด, `vulhub/log4j:2.8.2` ไม่มีบน Hub | เดาล่วงหน้าไม่ได้ ต้อง **probe ก่อนใช้** |
| **B — protocol/policy เปลี่ยน** | Maven Central บังคับ HTTPS, `apt-key` deprecated | เป็น **known-list** ของ workaround ที่สะสมได้ |

ทั้งสองแบบ automate เต็ม 100% ไม่ได้ (เดาไม่ได้ว่า Hub จะถอด image ไหนวันไหน)
เป้าคือ **detect เร็ว + แก้แบบมีโครงสร้าง** แทนการ debug สดหน้าเทอร์มินัล


## 4. คำถามที่เคลียร์แล้ว: ทำไมไม่ `apt install` แก้ในคอนเทนเนอร์เลย?

**ตอบ: ไม่ได้ ด้วย 2 เหตุผล**

**(ก) ปัญหาส่วนใหญ่เกิดก่อน container จะขึ้นมาด้วยซ้ำ** — มันพังตอน `docker pull` / `docker build`

| ปัญหา | apt install ช่วยไหม |
|---|---|
| base image หายจาก Hub | ❌ ยังไม่มี container ให้ install |
| Maven Central ปิด HTTP | ❌ เป็น policy ของ external server |
| disk เต็ม | ❌ ปัญหาฝั่ง host |

**(ข) [สำคัญต่อ thesis] ห้าม container อัพเดทตัวเอง** — testbed ต้องล็อกเวอร์ชันตายตัว
ถ้า container สั่ง upgrade เอง แล้ว dependency อื่น (glibc/openssl) ขยับ → ผล exploit อาจเปลี่ยนโดยไม่รู้ว่าเปลี่ยนเพราะ log4j หรือเพราะอย่างอื่น = **ไม่ reproducible** ซึ่งขัด core thesis ตรงๆ

**สรุปหลักการ: แยก 2 ชั้น**
- **Build-time** (นอก container, แก้ *recipe*/Dockerfile ได้) → เช่น `openjdk` → `eclipse-temurin`
- **Runtime** (ใน container ตอน exploit) → **ล็อกตายตัว ห้าม apt update/upgrade**

จุดที่พังคือ "ตอนสร้าง" ไม่ใช่ "ตอนรัน" → automation ที่ถูกคือ **pre-flight check ก่อน build** ไม่ใช่ให้ container ทำเอง


## 5. แนวทางที่เลือก — Hybrid source ต่อ CVE

ไม่พึ่ง Vulhub อย่างเดียว เพราะมันไม่มี patched image ให้ และถูกถอดเมื่อไหร่ก็ได้

| State | ดึงจากไหน | เหตุผล |
|---|---|---|
| **Before (vulnerable)** | Vulhub ตรงๆ | เป็น recipe ที่ community validate attack scenario แล้ว ไม่ต้อง reproduce เอง |
| **After (fixed)** | Central registry (Maven Central ฯลฯ) | immutable เก็บทุกเวอร์ชันถาวร ตัดปัญหา Type A ฝั่ง app-dependency |

**หลักคิด:** เปลี่ยนจาก "ใช้ของสำเร็จรูปที่คนอื่นคุมวงจรชีวิต" → "ประกอบเองจาก layer ที่รู้ policy การเก็บรักษาแน่ชัด"


## 6. การ resolve dependency — ใช้ CPE + FixedVersion จาก Trivy

**ไม่ทำ mapping table ต่อ CVE เอง** (จะเป็นภาระดูแลทุกครั้งที่เพิ่ม CVE)
แต่อ่าน **CPE ที่ Trivy ให้มาอยู่แล้ว** + **FixedVersion** เป็นตัวระบุ product/เวอร์ชัน

```
cpe:2.3:a:<vendor>:<product>:<version>:...
        apache   log4j    2.8.1
```

```
Trivy scan → ได้ CPE(vendor,product,version) + FixedVersion
   └── รู้ว่าเป็น product อะไร เวอร์ชันไหนพัง เวอร์ชันไหนหาย
         └── resolver ดึง fixed version จาก central registry ที่ตรง product
```

**ข้อดี:** rule เชื่อมอยู่ระดับ *product* (log4j/struts/tomcat) ไม่ใช่ระดับ CVE
→ จำนวน entry น้อยกว่าจำนวน CVE มาก (สอดคล้องแนวคิด group by CPE ลด duplication)

**ข้อจำกัดที่ยังเหลือ:** CPE เป็น naming ของ NVD ไม่ใช่ของ Maven
→ ยังต้องมี field เชื่อม vendor/product → Maven coordinate (group_id/artifact_id) อยู่ดี
แต่เป็นงาน one-time ต่อ *product* ไม่ใช่ต่อ CVE


## 7. ⚠️ ปัญหาที่ยังเปิดอยู่ (สำคัญ — อ่านก่อนเขียนโค้ด)

จุดที่ไล่คิดแล้วว่า design ปัจจุบันยังไม่ครอบคลุม:

### 7.1 สมมติฐาน "fix = อัพเวอร์ชัน library" ไม่จริงเสมอ  🔴 กระทบ thesis
- **CVE-2020-1938 (Ghostcat)** — fix จริงคือ **config** (`secretRequired`, ปิด/bind AJP) + Tomcat เป็น base image เอง ไม่ใช่ jar → resolver สาย Maven ใช้ไม่ได้
- **CVE-2021-44228 (Log4Shell)** — mitigate ได้ด้วยลบ class `JndiLookup` โดยไม่ขยับเวอร์ชัน

**เช็คแล้วว่า Trivy บอก fix แบบ config ได้ไหม → ไม่ได้**
Trivy เป็น SCA scanner โมเดลมันเป็น version-based ล้วน มีแค่ `FixedVersion` ไม่มี field mitigation/workaround
ข้อมูล config-fix อยู่ใน NVD References / CISA KEV `requiredAction` / vendor advisory — แต่เป็น **text ที่คนต้องอ่าน** parse ตรงๆ ไม่ได้ → ถ้าจะเอาเข้า pipeline ต้อง manual encode ต่อ CVE อยู่ดี

### 7.2 artifact ที่ถูกช่องโหว่อาจไม่ใช่ตัว core  🟡 ต้องดู output จริง
- **CVE-2017-9805 (S2-052)** ช่องโหว่อยู่ที่ `struts2-rest-plugin` ไม่ใช่ `struts2-core`
- **ต้องดู Trivy output จริงว่ามันแยก artifact ละเอียดแค่ไหน** — ถ้า Trivy รายงาน CPE ระดับ artifact ถูกต้อง (ชี้ `rest-plugin` ตรงๆ) + resolver เอา FixedVersion ไป inject ตรง artifact นั้น → **ไม่เป็นปัญหา**
- ยังไม่ควรเดา ต้องเห็นของจริงก่อน

### 7.3 บั๊มเวอร์ชัน Struts อาจลาก transitive dep พัง / ไม่ fix จริง  🟡
struts2-core ขยับเวอร์ชันมักลาก XWork/OGNL/commons-* ที่มี constraint เอง
→ อาจ build พัง หรือ build ผ่านแต่ dependency tree ยังมีตัวเก่าปน (คิดว่า patch แล้วแต่ยังไม่)

### 7.4 แยกไม่ออก: "FixedVersion ผิด" vs "fix ไม่ได้ผล"  🔴 กระทบ thesis — ยังไม่มีคำตอบ
ถ้า re-exploit หลัง patch **สำเร็จ** จะตีความว่าอะไร?
- FixedVersion บอกผิด (ควรเป็นเวอร์ชันอื่น)? หรือ
- FixedVersion ถูกแต่ fix ไม่ได้ผลจริง?

สองอันนี้คือข้อสรุปคนละแบบ แต่ automation ตอนนี้แยกไม่ออก → ต้องออกแบบ **state แยก**
เพิ่มเติม: Trivy บางทีให้ FixedVersion มา **หลายค่า/หลาย branch** (เช่น backport 2.12.2 กับ 2.16.0) → resolver ต้องมี policy ว่าเลือกอันไหน (ยังไม่มี)
→ *ยังไม่ต้องแก้ตอนนี้ แต่จะโผล่ตอน re-exploit สำเร็จครั้งแรก ค่อยออกแบบ state ตอนนั้นทัน*

### 7.5 Hybrid ยังไม่ปิด Type A สนิท  🟡
Maven Central immutable จริง แต่ Dockerfile ยังต้องมี base image (`eclipse-temurin`) บน Docker Hub ที่ไม่รับประกันเก็บทุก tag ตลอดกาล
→ drift ลดลงแต่ไม่หาย แกน base image ยังพังได้ (เหมือนที่ `openjdk` หายเป๊ะ) → ต้อง scope


## 8. Scope เฟสแรก (ตกลงแล้ว)

เข้าเฟสแรกได้เฉพาะ CVE ที่ครบ **ทั้ง 3 เงื่อนไข**:

1. ✅ base image ยัง pull ได้
2. ✅ dependency อยู่ registry ที่ immutable (Maven Central / PyPI / crates.io)
3. ✅ fix เป็น **version bump** (ไม่ใช่ config-based)

ขาดข้อใดข้อหนึ่ง → เข้ากอง "เฟสหลัง" ไปก่อน (เช่น Ghostcat ที่ fix เป็น config น่าจะตกเฟสหลัง)

> การที่ version-based scanner (Trivy) มองไม่เห็น config-based fix
> **เขียนเป็น limitation / contribution ใน thesis ได้เลย** — เป็นอีกมุมที่ static score จับไม่ได้


## 9. Next step ที่เป็นรูปธรรมสุด

**ดู Trivy output จริงของ 4 CVE ที่เหลือ** (Log4Shell, Struts2 S2-045/5638, S2-052/9805, Tomcat AJP)
เพราะมันตอบได้ทีเดียวหลายข้อ:

- ตอบ **7.2** — Trivy แยก artifact (`core` vs `rest-plugin`) ละเอียดไหม
- ตอบ **7.5 / scope** — base image ตัวไหนยัง pull ได้ / dependency ตัวไหน immutable
- ตอบ **7.1** — ตัวไหน fix ด้วย version bump ได้ / ตัวไหนเป็น config → คัดเข้าเฟสแรก vs เฟสหลัง

**เลิกเดา → ดูของจริง** แล้วค่อยร่าง schema + resolver ตามที่เห็นจริง

---
*หมายเหตุ: ทั้งหมดนี้ยังเป็นขั้น design ยังไม่มีโค้ด automation จริง — ตั้งใจไว้ว่าจะเริ่มเขียนหลังเห็น Trivy output ของ CVE ที่เหลือ*
